In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc gem-suite matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Nishimoriho fázový přechod

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Odhadovaná spotřeba: 3 minuty na procesoru Heron r2 (POZNÁMKA: Jedná se pouze o odhad. Skutečná doba běhu se může lišit.)*
## Výsledky učení
Po absolvování tohoto tutoriálu by uživatelé měli dosáhnout následujících výsledků:
- Porozumět Nishimoriho fázovému přechodu a tomu, jak se projevuje jako výskyt dlouhodosahového provázání v náhodně svázaném Isingově modelu.
- Implementovat protokol *generování provázání měřením* (GEM) na kvantovém hardwaru pomocí měření uprostřed obvodu a obvodů konstantní hloubky.
- Charakterizovat přechod extrakcí dvoubodové korelace a normalizovaného rozptylu magnetizace z experimentálních dat.

## Předpoklady
Před zahájením tohoto tutoriálu doporučujeme znalost následujících témat:
- Průvodce [Měření qubitů](/guides/measure-qubits), zejména část o měření uprostřed obvodu, na kterém protokol GEM závisí.
- [Přesná a zašuměná simulace s primitixy Qiskit Aer](/guides/simulate-with-qiskit-aer), pomocí nichž je spouštěna sekce malého rozsahu.
- [Dlouhodosahové provázání s dynamickými obvody](/tutorials/long-range-entanglement), doplňkový tutoriál využívající stejné paradigma provázání pomocí měření.
- [Mřížka heavy-hex](https://www.ibm.com/quantum/blog/heavy-hex-lattice), topologie hardwaru IBM&reg;, na níž je postavena plaketová mřížka.
## Pozadí
Tento tutoriál ukazuje, jak realizovat Nishimoriho fázový přechod na kvantovém procesoru. Tento experiment byl původně popsán v článku [*Realizing the Nishimori transition across the error threshold for constant-depth quantum circuits*](https://arxiv.org/abs/2309.02863).

Nishimoriho fázový přechod označuje přechod mezi krátkodosahovými a dlouhodosahovými uspořádanými fázemi v náhodně svázaném Isingově modelu. Na kvantovém počítači se fáze s dlouhodobým uspořádáním projevuje jako stav, ve kterém jsou qubity provázány přes celé zařízení. Tento vysoce provázaný stav se připravuje pomocí protokolu *generování provázání měřením* (GEM). Díky využití měření uprostřed obvodu je protokol GEM schopen provázat qubity přes celé zařízení pomocí obvodů pouze konstantní hloubky. Tento tutoriál využívá implementaci protokolu GEM ze softwarového balíčku [GEM Suite](https://github.com/qiskit-community/gem-suite).
## Požadavky
Před zahájením tohoto tutoriálu se ujisti, že máš nainstalováno následující:

- Qiskit SDK v1.0 nebo novější, s podporou [vizualizace](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 nebo novější (`pip install qiskit-ibm-runtime`)
- Qiskit Aer v0.14 nebo novější (`pip install qiskit-aer`)
- GEM Suite (`pip install gem-suite`)
## Nastavení

In [1]:
import matplotlib.pyplot as plt
import warnings

from collections import defaultdict

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator

from qiskit.transpiler import generate_preset_pass_manager

from gem_suite import PlaquetteLattice
from gem_suite.experiments import GemExperiment

## Příklad s malou simulací

V této části je celý postup ukázán na bezešumném simulátoru `AerSimulator`. Plaketová mřížka je omezena na jednu plaketu (12 qubitů), takže simulace zůstane malá a rychlá, přičemž stále procvičí každou část protokolu GEM: měření uprostřed obvodu, přechodový úhel $R_{ZZ}$, dekódování a analýzu normalizovaného rozptylu. Stejný postup je později použit pro více plaket a plnou mřížku na reálném hardwaru.

### Krok 1: Přeložení klasických vstupů na kvantový problém

Protokol GEM funguje na kvantovém procesoru, jehož konektivita qubitů je popsána mřížkou. Dnešní kvantové procesory IBM Quantum&reg; používají [mřížku heavy-hex](https://www.ibm.com/quantum/blog/heavy-hex-lattice). Qubity procesoru jsou seskupeny do *plaket* podle toho, které základní buňce mřížky patří. Protože qubit může patřit do více než jedné základní buňky, plakety nejsou vzájemně disjunktní. Na mřížce heavy-hex má plaket 12 qubitů. Samotné plakety také tvoří mřížku, přičemž dvě plakety jsou propojeny, pokud sdílejí některé qubity. Na mřížce heavy-hex sousední plakety sdílejí 3 qubity.

V softwarovém balíčku GEM Suite je základní třídou pro implementaci protokolu GEM třída `PlaquetteLattice`, která reprezentuje mřížku plaket (jež se liší od samotné mřížky heavy-hex). Objekt `PlaquetteLattice` lze inicializovat z mapy párování qubitů. Aktuálně jsou podporovány pouze mapy párování heavy-hex.

Následující buňka kódu inicializuje plaketovou mřížku z mapy párování kvantového procesoru (QPU). Plaketová mřížka ne vždy pokrývá celý hardware. Například procesor `ibm_torino` má celkem 133 qubitů, ale největší plaketová mřížka, která se na zařízení vejde, využívá pouze 125 z nich a zahrnuje 18 plaket; `ibm_pittsburgh` (156 qubitů) podobně vejde 144 qubitů do 21 plaket. Stejný vzor platí i pro ostatní QPU s architekturou heavy-hex s různým počtem qubitů.

In [ ]:
# QiskitRuntimeService.save_account(channel="ibm_quantum", token="<YOUR_API_KEY>", overwrite=True,
# set_as_default=True)
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
aer_backend = AerSimulator.from_backend(backend)
plaquette_lattice = PlaquetteLattice.from_coupling_map(backend.coupling_map)

print(f"Number of qubits in backend: {backend.num_qubits}")
print(
    f"Number of qubits in plaquette lattice: {len(list(plaquette_lattice.qubits()))}"
)
print(f"Number of plaquettes: {len(list(plaquette_lattice.plaquettes()))}")

Plaketovou mřížku lze vizualizovat vygenerováním diagramu jejího grafového zobrazení. V diagramu jsou plakety reprezentovány popsanými šestiúhelníky a dvě plakety jsou propojeny hranou, pokud sdílejí qubity.

In [3]:
plaquette_lattice.draw_plaquettes()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif)

Informace o jednotlivých plaketech, například o qubitech, které obsahují, lze získat pomocí metody `plaquettes`.

In [4]:
# Get a list of the plaquettes
plaquettes = list(plaquette_lattice.plaquettes())
# Display information about plaquette 0
plaquettes[0]

PyPlaquette(index=0, qubits=[3, 4, 5, 6, 7, 16, 17, 23, 24, 25, 26, 27], neighbors=[4, 3, 1])

You can also produce a diagram of the underlying qubits that form the plaquette lattice.

In [5]:
plaquette_lattice.draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif" alt="Output of the previous code cell" />

Také lze vytvořit diagram základních qubitů tvořících plaketovou mřížku.

In [6]:
# Filter the plaquette lattice down to a single plaquette (12 qubits)
# so the AerSimulator run stays fast. The full lattice is used later
# in the large-scale hardware example.
gem_exp = GemExperiment(plaquette_lattice.filter([9]), backend=aer_backend)

# visualize the plaquette lattice after filtering
plaquette_lattice.filter([9]).draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif)

Kromě popisků qubitů a hran označujících, které qubity jsou propojeny, diagram obsahuje tři další informace relevantní pro protokol GEM:
- Každý qubit je buď stínovaný (šedý), nebo nestínovaný. Stínované qubity jsou „site" qubity reprezentující místa Isingova modelu a nestínované qubity jsou „bond" qubity sloužící k zprostředkování interakcí mezi site qubity.
- Každý site qubit je označen buď (A), nebo (B), přičemž to označuje jednu ze dvou rolí, které může site qubit v protokolu GEM hrát (role jsou vysvětleny dále).
- Každá hrana je obarvena jednou ze šesti barev, čímž jsou hrany rozděleny do šesti skupin. Toto rozdělení určuje, jak lze paralelizovat dvoiqubitové brány, a také různé plánovací vzory, které pravděpodobně způsobí různé množství chyb na zašuměném kvantovém procesoru. Protože hrany ve skupině jsou disjunktní, lze vrstvu dvoiqubitových bran aplikovat na tyto hrany současně. Ve skutečnosti je možné rozdělit šest barev do tří skupin po dvou barvách tak, aby sjednocení každé skupiny dvou barev bylo stále disjunktní. Proto jsou ke aktivaci každé hrany potřeba pouze tři vrstvy dvoiqubitových bran. Existuje 12 způsobů, jak takto rozdělit šest barev, a každé takové rozdělení poskytuje jiný 3-vrstvý plán bran.

Nyní, když jsi vytvořil plaketovou mřížku, dalším krokem je inicializace objektu `GemExperiment` předáním jak plaketové mřížky, tak backendu, na kterém chceš experiment spustit. Třída `GemExperiment` spravuje skutečnou implementaci protokolu GEM, včetně generování obvodů, odesílání úloh a analýzy dat. Následující buňka kódu inicializuje třídu experimentu a zároveň omezuje plaketovou mřížku na jednu plaketu (12 qubitů), čímž simulace zůstane malá a rychlá. Plná plaketová mřížka je použita později při škálování na reálný hardware.

In [7]:
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

Total number of circuits: 252


![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif)

Obvod protokolu GEM je sestaven pomocí následujících kroků:
1. Připrav stav všech $|+\rangle$ aplikováním Hadamardovy brány na každý qubit.
2. Aplikuj bránu $R_{ZZ}$ mezi každým párem propojených qubitů. Toho lze dosáhnout pomocí tří vrstev bran. Každá brána $R_{ZZ}$ působí na site qubit a bond qubit. Pokud je site qubit označen (B), pak je úhel pevně nastaven na $\frac{\pi}{2}$. Pokud je site qubit označen (A), pak je úhel volný a generuje různé obvody. Ve výchozím nastavení je rozsah úhlů nastaven na 21 rovnoměrně rozmístěných bodů mezi $0$ a $\frac{\pi}{2}$ včetně.
3. Změř každý bond qubit v Pauliho bázi $X$. Protože qubity jsou měřeny v Pauliho bázi $Z$, toho lze dosáhnout aplikováním Hadamardovy brány před měřením qubitu.

Všimni si, že článek citovaný v úvodu tohoto tutoriálu používá jinou konvenci pro úhel $R_{ZZ}$, která se od konvence použité v tomto tutoriálu liší faktorem 2.

V kroku 3 jsou měřeny pouze bond qubity. Abychom pochopili, v jakém stavu zůstávají site qubity, je užitečné uvažovat případ, kdy je úhel $R_{ZZ}$ aplikovaný na site qubity (A) v kroku 2 roven $\frac{\pi}{2}$. V tomto případě jsou site qubity ponechány ve vysoce provázaném stavu podobném stavu GHZ,

$$
\lvert \text{GHZ} \rangle = \lvert 00 \cdots 00 \rangle + \lvert 11 \cdots 11 \rangle.
$$

Kvůli náhodnosti výsledků měření může být skutečný stav site qubitů jiným stavem s dlouhodobým uspořádáním, například $\lvert 00110 \rangle + \lvert 11001 \rangle$. Stav GHZ však lze obnovit aplikací dekódovací operace na základě výsledků měření. Když je úhel $R_{ZZ}$ snižován z $\frac{\pi}{2}$, dlouhodobé uspořádání lze stále obnovit až do kritického úhlu, který v nepřítomnosti šumu je přibližně $0.3 \pi$. Pod tímto úhlem výsledný stav již nevykazuje dlouhodosahové provázání. Tento přechod mezi přítomností a nepřítomností dlouhodobého uspořádání je Nishimoriho fázový přechod.

Ve výše uvedeném popisu byly site qubity ponechány neměřeny a dekódovací operaci lze provést aplikací kvantových bran. V experimentu implementovaném v GEM suite jsou site qubity ve skutečnosti měřeny a dekódovací operace je aplikována v klasickém post-procesním kroku.

Ve výše uvedeném popisu lze dekódovací operaci provést aplikací kvantových bran na site qubity k obnovení kvantového stavu. Pokud je však cílem okamžitě měřit stav (například pro účely charakterizace), pak lze site qubity měřit společně s bond qubity a dekódovací operaci aplikovat v klasickém post-procesním kroku.

Obvod protokolu GEM závisí kromě úhlu $R_{ZZ}$ v kroku 2, který ve výchozím nastavení přechází přes 21 hodnot, také na plánovacím vzoru použitém k implementaci tří vrstev bran $R_{ZZ}$. Jak bylo diskutováno dříve, existuje 12 takových plánovacích vzorů. Celkový počet obvodů v experimentu je tedy $21 \times 12 = 252$.

Obvody experimentu lze generovat pomocí metody `circuits` třídy `GemExperiment`.

In [8]:
# Restrict experiment to the first scheduling pattern
gem_exp.set_experiment_options(schedule_idx=0)

# There are less circuits now
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

# Print the RZZ angles swept over
print(f"RZZ angles:\n{gem_exp.parameters()}")

Total number of circuits: 21
RZZ angles:
[0.         0.07853982 0.15707963 0.23561945 0.31415927 0.39269908
 0.4712389  0.54977871 0.62831853 0.70685835 0.78539816 0.86393798
 0.9424778  1.02101761 1.09955743 1.17809725 1.25663706 1.33517688
 1.41371669 1.49225651 1.57079633]


The following code cell draws a diagram of the circuit at index 5. To reduce the size of the diagram, the measurement gates at the end of the circuit are removed.

In [9]:
# Get the circuit at index 5
circuit = circuits[5]
# Remove the final measurements to ease visualization
circuit.remove_final_measurements()
# Draw the circuit
circuit.draw("mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif" alt="Output of the previous code cell" />

Pro účely tohoto tutoriálu stačí uvažovat pouze jeden plánovací vzor. Následující buňka kódu omezuje experiment na první plánovací vzor. Výsledkem je, že experiment má pouze 21 obvodů, jeden pro každý přechodový úhel $R_{ZZ}$.

In [10]:
# Demonstrate setting transpile options
gem_exp.set_transpile_options(
    optimization_level=1  # This is the default optimization level
)
pass_manager = generate_preset_pass_manager(
    backend=aer_backend,
    initial_layout=list(gem_exp.physical_qubits),
    **dict(gem_exp.transpile_options),
)
transpiled = pass_manager.run(circuit)
transpiled.draw("mpl", idle_wires=False, fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

To execute the GEM protocol circuits on the hardware, call the `run` method of the `GemExperiment` object. You can specify the number of shots you want to sample from each circuit. The `run` method returns an [ExperimentData](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.framework.ExperimentData.html) object which you should save to a variable. Note that the `run` method only submits jobs without waiting for them to finish, so it is a non-blocking call.

In [11]:
exp_data = gem_exp.run(shots=10_000)

Následující buňka kódu vykreslí diagram obvodu na indexu 5. Aby se zmenšila velikost diagramu, jsou měřicí brány na konci obvodu odstraněny.

In [12]:
# The noiseless AerSimulator produces zero-variance UFloat objects in the
# analysis, which triggers a harmless warning from the `uncertainties`
# library. Suppress it so the output stays clean.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", message="Using UFloat objects with std_dev==0"
    )
    exp_data.block_for_results()
exp_data

ExperimentData(GemExperiment, 90bf2a90-f729-4c4e-a6da-664aecb11039, job_ids=['04a7c405-47fd-46ca-aa4b-aaf7e339cfbe'], metadata=<5 items>, figure_names=['two_point_correlation.svg', 'normalized_variance.svg', 'plaquette_ops.svg', 'bond_ops.svg'])

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif)

### Krok 2: Optimalizace problému pro spuštění na kvantovém hardwaru
Transpilace kvantových obvodů pro spuštění na hardwaru obvykle zahrnuje [několik fází](/guides/transpiler-stages). Typicky jsou fáze s největší výpočetní náročností výběr rozmístění qubitů, směrování dvoiqubitových bran podle konektivity qubitů v hardwaru a optimalizace obvodu pro minimalizaci počtu bran a hloubky. V protokolu GEM jsou fáze rozmístění a směrování zbytečné, protože konektivita hardwaru je již zahrnuta v návrhu protokolu. Obvody již mají rozmístění qubitů a dvoiqubitové brány jsou již namapovány na nativní spojení. Navíc, aby byla zachována struktura obvodu při změně úhlu $R_{ZZ}$, měla by být prováděna pouze velmi základní optimalizace obvodu.

Třída `GemExperiment` transparentně transpiluje obvody při provádění experimentu. Fáze rozmístění a směrování jsou již ve výchozím nastavení přepsány tak, aby nic nedělaly, a optimalizace obvodu je prováděna na úrovni, která optimalizuje pouze jednoqubitové brány. Přesto můžeš přepsat nebo předat další možnosti pomocí metody `set_transpile_options`. Pro účely vizualizace následující buňka kódu ručně transpiluje dříve zobrazený obvod a vykreslí transpilovaný obvod.

In [13]:
def magnetization_distribution(
    counts_dict: dict[str, int],
) -> dict[str, float]:
    """Compute magnetization distribution from counts dictionary."""
    # Construct dictionary from magnetization to count
    mag_dist = defaultdict(float)
    for bitstring, count in counts_dict.items():
        mag = bitstring.count("0") - bitstring.count("1")
        mag_dist[mag] += count
    # Normalize
    shots = sum(counts_dict.values())
    for mag in mag_dist:
        mag_dist[mag] /= shots
    return mag_dist


# Get counts dictionaries with and without decoding
data = exp_data.data()
# Get the last data point, which is at the angle for the GHZ state
raw_counts = data[-1]["counts"]
# Without decoding
site_indices = [
    i for i, q in enumerate(gem_exp.plaquettes.qubits()) if q.role == "Site"
]
site_raw_counts = defaultdict(int)
for key, val in raw_counts.items():
    site_str = "".join(key[-1 - i] for i in site_indices)
    site_raw_counts[site_str] += val
# With decoding
_, site_decoded_counts = gem_exp.plaquettes.decode_outcomes(
    raw_counts, return_counts=True
)

# Compute magnetization distribution
raw_magnetization = magnetization_distribution(site_raw_counts)
decoded_magnetization = magnetization_distribution(site_decoded_counts)

# Plot
plt.bar(*zip(*raw_magnetization.items()), label="raw")
plt.bar(*zip(*decoded_magnetization.items()), label="decoded", width=0.3)
plt.legend()
plt.xlabel("Magnetization")
plt.ylabel("Frequency")
plt.title("Magnetization distribution with and without decoding")

Text(0.5, 1.0, 'Magnetization distribution with and without decoding')

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/8ead3582-16df-4616-836c-bdce867ad6b8-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif)

### Krok 3: Spuštění pomocí primitiv Qiskit
Pro spuštění obvodů protokolu GEM na hardwaru zavolej metodu `run` objektu `GemExperiment`. Můžeš určit počet vzorků, které chceš odebrat z každého obvodu. Metoda `run` vrací objekt [ExperimentData](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.framework.ExperimentData.html), který bys měl uložit do proměnné. Všimni si, že metoda `run` pouze odesílá úlohy bez čekání na jejich dokončení, jde tedy o neblokující volání.

In [14]:
exp_data.figure("two_point_correlation")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/4ecb25c8-e572-49af-a879-9943039db131-0.avif" alt="Output of the previous code cell" />

Pro čekání na výsledky zavolej metodu `block_for_results` objektu `ExperimentData`. Toto volání způsobí, že interpret počká, dokud nejsou úlohy dokončeny.

In [15]:
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/2b351d68-3924-445a-94ef-047b16214e8a-0.avif" alt="Output of the previous code cell" />

## Large-scale hardware example

Having validated the protocol on a simulator, you can now scale up the experiment and run it on the real quantum hardware backend selected in the [Setup](#setup) section. This example uses two larger problem sizes:

- **Six plaquettes (~49 qubits)**: a mid-size run that already shows the rightward shift of the critical point under hardware noise.
- **The full plaquette lattice**: every plaquette the device's heavy-hex topology supports (for example, 18 plaquettes / 125 qubits on `ibm_torino` or 21 plaquettes / 144 qubits on `ibm_pittsburgh`), entangling qubits across the entire device with constant-depth circuits.

The single code cell below is self-contained: it builds the plaquette lattice from the backend's coupling map and runs both experiments, so this section can be executed after the [Setup](#setup) cells without first running the small-scale section.

In [ ]:
# -------------------------Step 1-------------------------
# Initialize the runtime service, pick a real quantum hardware backend,
# and build the plaquette lattice from its coupling map. This is repeated
# from the small-scale example so this cell can run standalone after the
# Setup section. The full plaquette lattice is the "large-scale" target;
# a six-plaquette subset (range(3, 9)) is also used to show an intermediate
# scaling step.
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
plaquette_lattice = PlaquetteLattice.from_coupling_map(backend.coupling_map)

# Build a GemExperiment for the full plaquette lattice and one for the
# six-plaquette subset, each restricted to a single scheduling pattern so
# the experiment has one circuit per RZZ angle (21 circuits total).
gem_exp_full = GemExperiment(plaquette_lattice, backend=backend)
gem_exp_full.set_experiment_options(schedule_idx=0)
gem_exp_6 = GemExperiment(
    plaquette_lattice.filter(range(3, 9)), backend=backend
)
gem_exp_6.set_experiment_options(schedule_idx=0)

circuits = gem_exp_full.circuits()
print(f"Total number of circuits (full lattice): {len(circuits)}")

# -------------------------Step 2-------------------------
# GemExperiment transpiles internally for the target backend: the layout
# and routing stages are overridden because the plaquette lattice already
# matches the hardware connectivity, and optimization is restricted so the
# RZZ angle structure is preserved. The code below manually transpiles one
# circuit from the six-plaquette experiment with the same settings this
# experiment will use, and draws it for inspection. (The full-lattice
# transpiled circuit has too many qubits to visualize cleanly, so the
# six-plaquette circuit is used here as a representative example.)
gem_exp_6.set_transpile_options(optimization_level=1)
circuits_6 = gem_exp_6.circuits()
pass_manager = generate_preset_pass_manager(
    backend=backend,
    initial_layout=list(gem_exp_6.physical_qubits),
    **dict(gem_exp_6.transpile_options),
)
transpiled = pass_manager.run(circuits_6[5])
display(transpiled.draw("mpl", idle_wires=False, fold=-1, scale=0.5))

# -------------------------Step 3-------------------------
# Run both problem sizes on real hardware:
#   1. Six plaquettes (~49 qubits) — an intermediate scale-up.
#   2. The full plaquette lattice — every plaquette the device supports.
exp_data_6 = gem_exp_6.run(shots=10_000, job_tags=["TUT_NPT"])
exp_data_full = gem_exp_full.run(shots=10_000, job_tags=["TUT_NPT"])
exp_data_6.block_for_results()
exp_data_full.block_for_results()

# -------------------------Step 4-------------------------
# Plot the normalized variance at each scale. The peak marks the critical
# point of the Nishimori transition; as the system grows, hardware noise
# shifts the peak rightward.
display(exp_data_6.figure("normalized_variance"))
exp_data_full.figure("normalized_variance")

Total number of circuits (full lattice): 21


<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-1.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-2.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-3.avif" alt="Output of the previous code cell" />

### Krok 4: Post-processing a vrácení výsledku v požadovaném klasickém formátu
Při úhlu $R_{ZZ}$ rovném $\frac{\pi}{2}$ by byl dekódovaný stav v nepřítomnosti šumu stavem GHZ. Dlouhodobé uspořádání stavu GHZ lze vizualizovat vykreslením magnetizace naměřených bitových řetězců. Magnetizace $M$ je definována jako součet jednoqubitových Pauliho operátorů $Z$,
$$
M = \sum_{j=1}^N Z_j,
$$
kde $N$ je počet site qubitů. Její hodnota pro bitový řetězec se rovná rozdílu počtu nul a počtu jedniček. Měření stavu GHZ dává stav samých nul nebo samých jedniček se stejnou pravděpodobností, takže magnetizace by byla $+N$ v polovině případů a $-N$ ve druhé polovině. V přítomnosti chyb způsobených šumem by se objevily i jiné hodnoty, ale pokud šum není příliš velký, distribuce by stále vykazovala špičky poblíž $+N$ a $-N$.

Pro surové bitové řetězce před dekódováním by distribuce magnetizace odpovídala distribuci rovnoměrně náhodných bitových řetězců v nepřítomnosti šumu.

Následující buňka kódu vykreslí magnetizaci surových bitových řetězců a dekódovaných bitových řetězců při úhlu $R_{ZZ}$ rovném $\frac{\pi}{2}$.